In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from utilsPlots import *


In [3]:
datadir = Path('../../datadir')

figdir = Path('./figs')
figdir.mkdir(exist_ok=True, parents=True)


In [4]:
def all_the_stats(df):
    df = df.copy()
    df['age_decade'] = df.age // 10 * 10
    df['age_decade'] = df['age_decade'].apply(lambda x: f'{int(x)}-{int(x)+9}'
                                              if not np.isnan(x) else 'nan')

    def bmi_grouper(bmi):
        if bmi < 18.5:
            return 'a <18.5'
        elif bmi < 25:
            return 'b 18.5-25'
        elif bmi <= 30:
            return 'c 25-30'
        else:
            return 'd >30'
    df['bmi'] = df.weight / df.height**2
    df['bmi_group'] = df['bmi'].apply(bmi_grouper)

    print(df.ID.nunique(), 'unique participants')
    print(len(df), 'sessions')
    print()

    print('\033[1m' 'Repetitions' '\033[0m')
    temp = df.groupby('ID').count().groupby('day').count()['weight'].reset_index()
    print(temp.to_string(index=False, header = False))
    print()

    print('\033[1m' 'Type' '\033[0m')
    temp = df.groupby('type', dropna=False)['ID'].nunique().reset_index()
    print(temp.to_string(index=False, header = False))
    print()

    print('\033[1m' 'Location' '\033[0m')
    temp = df.groupby('location', dropna=False)['ID'].nunique().reset_index()
    print(temp.to_string(index=False, header = False))
    print()

    print('\033[1m' 'Sex' '\033[0m')
    temp = df.groupby('sex', dropna=False)['ID'].nunique().reset_index()
    print(temp.to_string(index=False, header = False))
    print()

    print('\033[1m' 'Age' '\033[0m')
    temp = df.groupby('age_decade', dropna=False)['ID'].nunique()
    print(temp.to_string(index=True, header = False))
    print(f'µ(σ)\t{df.age.mean():0.1f} ({df.age.std():0.1f})')
    print()

    print('\033[1m' 'BMI' '\033[0m')
    temp = df.groupby('bmi_group', dropna=False)['ID'].nunique()
    print(temp.to_string(index=True, header = False))
    print(f'µ(σ)\t{df.bmi.mean():0.1f} ({df.bmi.std():0.1f})')
    print()


In [5]:
df_info = pd.read_csv(datadir / 'participant_info.csv')
df_loc = pd.read_csv(datadir / 'locations.csv')
df_feat = pd.read_csv(datadir / 'video_features.csv')
df_all = df_info.merge(df_loc, on=['ID', 'day'])
df_all = df_all.merge(df_feat, on=['ID', 'day'])
df_all.sort_values(['ID', 'day'], inplace=True)

# df_all = df_all.drop_duplicates(subset='ID', keep='last')

df_trt = df_all[df_all.day.isin([118, 119])]
df_trt = df_trt[df_trt.duplicated('ID', keep=False)]
df_rest = df_all[~df_all.ID.isin(df_trt.ID)]
df_rest = df_rest.drop_duplicates(subset='ID', keep='last')
df_all = pd.concat([df_rest, df_trt])
df_nodup = df_all.drop_duplicates(subset='ID', keep='first')

df_all.to_csv('df_all.csv', index=False)


In [9]:
all_the_stats(df_nodup)


129 unique participants
129 sessions

Repetitions
1 129

Type
  DM 58
FSHD 28
 TYP 43

Location
     Community events 30
High-throughput study 94
 Neuromuscular clinic  5

Sex
M 72
W 57

Age
10-19     4
20-29    22
30-39    32
40-49    25
50-59    23
60-69    16
70-79     5
80-89     2
µ(σ)	43.4 (15.6)

BMI
a <18.5       8
b 18.5-25    65
c 25-30      33
d >30        23
µ(σ)	25.2 (5.5)



In [97]:
all_the_stats(df_all[df_all.type=='FSHD'])


28 unique participants
28 sessions

Repetitions
1 28

Type
FSHD 28

Location
     Community events  1
High-throughput study 26
 Neuromuscular clinic  1

Sex
M 24
W  4

Age
20-29    4
30-39    6
40-49    3
50-59    7
60-69    6
70-79    1
80-89    1
µ(σ)	48.2 (16.0)

BMI
a <18.5       1
b 18.5-25    13
c 25-30      10
d >30         4
µ(σ)	25.8 (4.6)



In [100]:
all_the_stats(df_all[df_all.type=='DM'])


58 unique participants
58 sessions

Repetitions
1 58

Type
DM 58

Location
     Community events 27
High-throughput study 27
 Neuromuscular clinic  4

Sex
M 27
W 31

Age
10-19     2
20-29    10
30-39    16
40-49    16
50-59     7
60-69     4
70-79     3
µ(σ)	41.1 (14.4)

BMI
a <18.5       6
b 18.5-25    27
c 25-30      11
d >30        14
µ(σ)	25.4 (6.6)



In [101]:
all_the_stats(df_all[df_all.type=='TYP'])


43 unique participants
43 sessions

Repetitions
1 43

Type
TYP 43

Location
     Community events  2
High-throughput study 41

Sex
M 21
W 22

Age
10-19     2
20-29     8
30-39    10
40-49     6
50-59     9
60-69     6
70-79     1
80-89     1
µ(σ)	43.5 (16.6)

BMI
a <18.5       1
b 18.5-25    25
c 25-30      12
d >30         5
µ(σ)	24.7 (4.4)

